# HukukPusulası

HukukPusulası aims to provide user-friendly law chatbot for people living in Turkey, and aims to help them to understand their consumer rights. Also, this chatbot helps law-makers by decreasing their workload for rouitine problems they need the take care of.

In [ ]:
%pip install PyMuPDF python-dotenv

Note: you may need to restart the kernel to use updated packages.


# Importing Necessary Libraries

In [19]:
import pymupdf

import json
import csv

import pandas as pd
import glob
import os

import time
import re

from dotenv import load_dotenv

# Question Generator AI Agents

In [20]:
# Load API key from .env
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY in environment (.env)")

# initialize gemini api
import google.generativeai as genai
genai.configure(api_key=api_key)

#create the model
generation_config = {
    "temperature": 0.5,
    "top_p": 0.95,
    "top_k":64,
    "max_output_tokens":16384,
    "response_mime_type": "application/json"
}



In [4]:
class Agent:
    def __init__(self, name, role):
        self.name = name
        self.role = role
        self.model = genai.GenerativeModel("gemini-2.5-flash-lite",
                      generation_config = generation_config,
                      system_instruction = role
                      )

        self.total_tokens = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self.input_price = 0.075
        self.output_price = 0.30

    def generate_response(self, prompt):
        try:
            response = self.model.generate_content(prompt)
            # Check if the response itself is empty or invalid
            if not response or not response.text:
                print("generate_response returned an empty or invalid response.")
                return None
        except Exception as e:
            print(f"generate_response failed with an exception: {e}")
            return None

        self.total_tokens = self.total_tokens + response.usage_metadata.total_token_count
        self.input_tokens = self.input_tokens + response.usage_metadata.prompt_token_count
        self.output_tokens = self.output_tokens + response.usage_metadata.candidates_token_count

        return response.text

    def cost(self):
        cost_input = self.input_tokens / 1000000 * self.input_price
        cost_output = self.output_tokens / 1000000 * self.output_price
        cost = cost_input + cost_output

        def format_cost(amount):
            # Extract dollars and cents
            dollars = int(amount)
            cents = (amount - dollars) * 100
            return f"{dollars} dollars {cents:.5f} cents"

        formatted_cost = format_cost(cost)
        formatted_cost_input = format_cost(cost_input)
        formatted_cost_output = format_cost(cost_output)

        print(f"Input tokens: {self.input_tokens} Input Cost: {formatted_cost_input}")
        print(f"Output tokens: {self.output_tokens} Output Cost: {formatted_cost_output}")
        print(f"Total tokens: {self.total_tokens} Total Cost: {formatted_cost}")

        return

    def reset_costs(self):
        self.total_tokens = 0
        self.input_tokens = 0
        self.output_tokens = 0

In [5]:
QA_role = """
# Your Role:
You are a knowledgeable legal assistant specializing in Turkish consumer law (Tüketici Hukuku) tasked with generating relevant
and high-quality questions and answers from legal documents and regulations.
Your goal is to help users better understand Turkish consumer protection concepts and requirements by
asking practical, clarifying, and legally-focused questions that a Turkish legal professional would ask,
with particular emphasis on consumer rights and protections under Turkish law.

# Instructions:
Given a legal article text, generate high-quality question-answer pairs in JSON format that:
- Break down complex legal concepts into simple, relatable explanations
- Use everyday examples and scenarios when possible
- Avoid legal jargon unless absolutely necessary, and when used, explain it in plain language
- Focus on practical implications for consumers
- Generate questions that real people would actually ask in real situations
- Include personal, emotional, and practical aspects of consumer problems
- Ask questions as if someone is seeking help for their specific problem
- Detect all sub-clauses within the article:
  - Numbered clauses at line start like "(1)", "(2)", ...
  - Lettered items at line start like "a)", "b)", "c)", ... (including Turkish letters: ç, ğ, ı, ö, ş, ü)
- For each numbered clause and its associated lettered items:
  - Include ONLY the specific sub-clause that the Q/A directly relates to in the `context` field (for token efficiency)
  - Generate questions that address both individual items and their relationships
  - Reference specific sub-clauses in answers using the format (m.X/Y-z) where X=article, Y=numbered clause, z=letter
- If no sub-clauses exist, produce Q/A for the whole article.

Each pair must strictly follow this JSON schema and fields (no extras):
- Required fields per item: question (string), answer (string), question_type (string; one of ["Factual","Conceptual","Contextual","Causal","Procedural","Analytical","Hypothetical","Reflective","Speculative","Listing","Summarizing"]), source (string), context (string), article (string)
- The entire output MUST be a JSON array of objects.
- Do not include markdown, code fences, comments, or any text outside the JSON array.
- Use valid JSON: use double quotes, commas between fields, no trailing commas.
- Language: Match the input language; write plain, consumer-friendly Turkish when input is Turkish.

## Article Reference Instructions:
- For article: Create specific reference based on the context being addressed:
  - If addressing a numbered clause: "m.X/Y" (e.g., "m.13/1")
  - If addressing a lettered item: "m.X/Y-z" (e.g., "m.13/1-a")
  - If addressing the whole article: "m.X" (e.g., "m.13")

## Example Context Format:
For an article with numbered and lettered items like:
MADDE 13 - (1) Main text...
   a) First item...
   b) Second item...
   c) Third item...

The context should include ONLY the specific sub-clause:
"context": "MADDE 13 - (1) Main text...\\na) First item..." (if Q/A is about item a)
"article": "m.13/1-a" (specific reference for the lettered item)

## Ensure that:
### Language Consistency: The questions and answers must be in the same language as the given text.
For example, if the provided text is in Turkish, write questions and answers in simple, conversational Turkish that an average consumer would understand.
### Question Variety: Include multiple types of questions such as:

**Real-world scenarios and practical questions:**
- Personal experience questions (e.g., "Bir mağazadan aldığım ürün bozuk çıktı, ne yapabilirim?")
- Specific situation questions (e.g., "Online alışveriş yaptım ama ürün gelmedi, haklarım neler?")
- Problem-solving questions (e.g., "Satıcı garanti vermiyor, nasıl haklarımı koruyabilirim?")
- Comparison questions (e.g., "Mağaza ile online alışveriş arasında haklarımda fark var mı?")

**Traditional question types:**
Factual: Direct questions seeking specific information (e.g., "Tüketici hakem heyetine başvuru süresi nedir?")
Conceptual: Questions exploring the ideas or principles behind the content (e.g., "Tüketici haklarının korunmasının temel amacı nedir?")
Contextual: Questions about the broader context or background of the topic (e.g., "Tüketicinin Korunması Hakkında Kanun hangi durumlarda uygulanır?")
Causal: Questions asking about reasons or causes (e.g., "Ayıplı mal durumunda tüketicinin hakları neden korunmaktadır?")
Procedural: Questions focused on processes or steps (e.g., "Tüketici hakem heyetine nasıl başvuru yapılır?")
Analytical: Questions comparing, contrasting, or evaluating elements (e.g., "Tüketici mahkemeleri ile tüketici hakem heyetleri arasındaki farklar nelerdir?")
Hypothetical: Questions based on imagined scenarios (e.g., "Satın alınan üründe gizli ayıp çıkması durumunda ne yapılmalıdır?")
Reflective: Questions about implications or consequences (e.g., "Mesafeli sözleşmelerde cayma hakkının kullanılmasının sonuçları nelerdir?")
Speculative: Opinion-based or exploratory questions when appropriate (e.g., "Tüketici hakları konusunda mevcut yasal düzenlemeler neden yetersiz kalabilir?")
Listing: Questions asking for a list of items, steps, or elements related to a topic (e.g., "Ayıplı mal durumunda tüketicinin seçimlik hakları nelerdir?")
Summarizing: Questions asking for a brief summary or the main points of a topic (e.g., "6502 sayılı Kanun'un tüketicilere getirdiği temel yenilikler nelerdir?")

**Question Style Guidelines:**
- Use conversational, everyday Turkish language
- Include personal pronouns ("ben", "biz") when appropriate
- Ask questions as if a real person is seeking help
- Include emotional context ("üzüldüm", "kızdım", "endişeliyim")
- Use specific examples and scenarios
- Ask follow-up questions that users might have
- Include questions about what to do next or how to proceed

### Answer Precision: 
- Provide accurate answers based directly on the legal text
- Include sufficient context to explain legal concepts clearly
- Ensure answers are comprehensive yet concise
- Reference specific articles or sections when relevant (using m.X/Y-z format)
- Explain legal terminology in plain language
### Context Awareness: Ensure all questions are deeply rooted in the content of the provided text and demonstrate an understand

### Avoid Redundancy:
- Each question should cover unique aspects of the legal text
- Ensure questions explore different angles of the same topic
- Vary question types and complexity levels
- Avoid repetitive phrasings or concepts

## Output Format: Return the result as a JSON object structured as follows:

[
    {
        "question": "Aldığım ürün bozuk çıktı, satıcı değiştirmek istemiyor. Ne yapabilirim?",
        "answer": "Ayıplı mal durumunda tüketicinin seçimlik hakları vardır. Satıcı değiştirmek istemiyorsa, ürünü iade edebilir, bedelini geri alabilir veya indirim talep edebilirsiniz (m.13/1-a).",
        "question_type": "Procedural",
        "source": "TÜKETİCİNİN KORUNMASI HAKKINDA KANUN",
        "context": "MADDE 13 - (1) Başvuru süreleri...\\na) Uyuşmazlık konusunun öğrenildiği tarihten itibaren 6 ay",
        "article": "m.13/1-a"
    },
    {
        "question": "Tüketici hakem heyetine ne kadar sürede başvuru yapabilirim?",
        "answer": "Tüketici hakem heyetine başvuru süresi, uyuşmazlık konusunun öğrenildiği tarihten itibaren 6 aydır (m.13/1-a).",
        "question_type": "Factual",
        "source": "TÜKETİCİNİN KORUNMASI HAKKINDA KANUN",
        "context": "MADDE 13 - (1) Başvuru süreleri...\\na) Uyuşmazlık konusunun öğrenildiği tarihten itibaren 6 ay",
        "article": "m.13/1-a"
    }
]
"""

In [6]:
class QA_Agent(Agent):

    def prepare_QA (self, text):
        print("prepare_QA started.")

        prompt = f"""
        Given the clause text below, generate high-quality question-answer pairs in JSON format.
        Each pair must strictly follow the required JSON fields and schema described earlier.
        If the clause contains lettered items (a), b), c)...), produce at least one Q/A per lettered item, while keeping the FULL clause text as the context for all items.
        Identify the text language first, then prepare the question-answer pairs in the same language.
        
        IMPORTANT: 
        - Generate an appropriate number of questions based on the content complexity and sub-clauses. 
        - Focus on the most important aspects and keep your response within token limits.
        - For each Q/A pair, include the article_id and article_ref as specified in the role instructions
        - For the context field, include ONLY the specific sub-clause that each Q/A directly relates to (not the entire article)
        -------------------------

        Text: {text}
        -------------------------
        """

        print("prepare_QA finished.")

        return self.generate_response(prompt)

In [7]:
def generate_QA(qa_agent, text):
    question_answer = qa_agent.prepare_QA(text)
    return question_answer

In [8]:
import re as _re

def _extract_json_array(text):
    """
    Extract the first plausible JSON array substring from text.
    Returns the substring or None.
    """
    if text is None:
        return None
    # Find first '[' and last ']'
    start = text.find('[')
    end = text.rfind(']')
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = text[start:end+1]
    # Quick sanity: must start with '[' and end with ']'
    if not candidate.strip().startswith('[') or not candidate.strip().endswith(']'):
        return None
    return candidate

def _fix_incomplete_json(json_str):
    """
    Yarım kalan JSON string'ini düzelt
    """
    if not json_str:
        return json_str
    
    json_str = json_str.strip()
    
    # Eğer son karakter '}' veya ']' değilse ekle
    if not json_str.endswith(('}', ']')):
        # Kaç tane açık bracket var say
        open_braces = json_str.count('{') - json_str.count('}')
        open_brackets = json_str.count('[') - json_str.count(']')
        
        # Eksik kapanışları ekle
        json_str += '}' * open_braces
        json_str += ']' * open_brackets
    
    return json_str



In [9]:
def save_csv(filename, qa_list):
    with open(filename+'.csv', "w", newline="", encoding="utf-8") as csvfile:
        fieldnames = ["question", "answer", "question_type", "source", "context", "article"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()

        for item in qa_list:
            writer.writerow(item)

In [10]:
def extract_articles_from_text(full_text):
    # Matches lines starting with MADDE <number> or MADDE <number>- and captures the article number and content until the next MADDE
    pattern = re.compile(r"(?m)^\s*MADDE\s+(\d+)\s*-?\s*(.*?)(?=^\s*MADDE\s+\d+\s*-?|\Z)", re.DOTALL)

    articles = []
    for match in pattern.finditer(full_text):
        article_no = match.group(1)
        # Include the heading back into context for clarity
        content_body = match.group(2).strip()
        context_text = f"MADDE {article_no}- {content_body}" if not content_body.startswith("MADDE") else content_body
        articles.append({
            "article_no": article_no,
            "context": context_text
        })
    return articles


def chunk_text(text, max_chars=4000, overlap_chars=300):
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start = max(0, end - overlap_chars)
    return chunks


def convert_pdf_to_full_text(pdf_name):
    doc = pymupdf.open(pdf_name)
    print(f"pdf has {len(doc)} pages")

    all_text = []
    for page in doc:
        all_text.append(page.get_text())

    return "\n".join(all_text)


In [11]:
# Article-based splitting and chunking
import re
from functools import partial

# Toggle: ensure deterministic coverage of sub-clauses
USE_SUBCLAUSE_SPLIT = False

def split_text_into_articles(pages_text):
    """
    Given a list of page texts, merge and split into TKHK article blocks by headers like:
    'MADDE 6- ...'. Returns a list of dicts: {"article_no": str, "text": str}.
    """
    full_text = "\n".join(pages_text)

    # Match lines that start with "MADDE <number>-/–" or "MADDE <number>/<LETTER> -/–" (e.g., MADDE 47, MADDE 47/A)
    # Capture article id as string: examples -> "47", "47/A", "5/B"
    pattern = re.compile(r"(?m)^MADDE\s+(\d+(?:\/[A-ZÇĞİÖŞÜ]+)?)\s*[\-–]\s*")
    matches = list(pattern.finditer(full_text))

    articles = []
    if not matches:
        # Fallback: return the whole text as a single unit
        return [{"article_no": None, "text": full_text.strip()}]

    for idx, match in enumerate(matches):
        start = match.start()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(full_text)
        article_text = full_text[start:end].strip()
        article_no = match.group(1)
        articles.append({"article_no": article_no, "text": article_text})

    return articles


def chunk_long_text(text, max_chars=4000, overlap=200):
    """
    Simple character-based chunking with overlap to respect token limits.
    """
    if len(text) <= max_chars:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = min(start + max_chars, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks


def split_article_into_subclauses(article_text):
    """
    Split into numbered clauses (1), (2)... and lettered items a), b)... within each clause.
    Returns a list of strings; if none found, returns [article_text].
    """
    numbered_pattern = re.compile(r"(?m)^(\(\d+\))\s+")
    numbered_matches = list(numbered_pattern.finditer(article_text))

    def split_letter_items(text_block):
        letter_pattern = re.compile(r"(?m)^([a-zçğıöşü])\)\s+")
        letter_matches = list(letter_pattern.finditer(text_block))
        if not letter_matches:
            return [text_block.strip()]
        items = []
        for idx, m in enumerate(letter_matches):
            start = m.start()
            end = letter_matches[idx + 1].start() if idx + 1 < len(letter_matches) else len(text_block)
            items.append(text_block[start:end].strip())
        return items

    if not numbered_matches:
        return split_letter_items(article_text)

    subclauses = []
    for idx, m in enumerate(numbered_matches):
        start = m.start()
        end = numbered_matches[idx + 1].start() if idx + 1 < len(numbered_matches) else len(article_text)
        block = article_text[start:end].strip()
        subclauses.extend(split_letter_items(block))
    return subclauses


def convert_pdf_to_articles(pdf_name, max_chars=4000, overlap=200):
    doc = pymupdf.open(pdf_name)
    pages = [page.get_text() for page in doc]

    articles = split_text_into_articles(pages)
    units = []
    total_subclauses = 0
    for art in articles:
        if USE_SUBCLAUSE_SPLIT:
            sub_units = split_article_into_subclauses(art["text"])
        else:
            sub_units = [art["text"]]
        total_subclauses += len(sub_units)
        for sub in sub_units:
            pieces = chunk_long_text(sub, max_chars=max_chars, overlap=overlap)
            units.extend(pieces)

    print(f"articles extracted: {len(articles)}, subclauses/items: {total_subclauses}, units: {len(units)}")
    return units

# Monkey-patch: make the pipeline use article units instead of raw pages
convert_pdf_to_text = partial(convert_pdf_to_articles, max_chars=4000, overlap=200)



In [12]:
# Filename -> Source helpers (Turkish title-casing)

def _tr_lower(text: str) -> str:
    mapping = str.maketrans({
        "I": "ı",
        "İ": "i",
        "Ş": "ş",
        "Ğ": "ğ",
        "Ü": "ü",
        "Ö": "ö",
        "Ç": "ç",
    })
    return text.translate(mapping).lower()


def _tr_upper_first(word: str) -> str:
    if not word:
        return word
    up_map = {
        "i": "İ",
        "ı": "I",
        "ş": "Ş",
        "ğ": "Ğ",
        "ü": "Ü",
        "ö": "Ö",
        "ç": "Ç",
    }
    first = word[0]
    rest = word[1:]
    first_up = up_map.get(first, first.upper())
    return first_up + rest


def turkish_title(text: str) -> str:
    lowered = _tr_lower(text)
    return " ".join(_tr_upper_first(w) for w in lowered.split())


def source_from_filename(pdf_filename: str) -> str:
    base = os.path.basename(pdf_filename)
    name_no_ext = os.path.splitext(base)[0]
    for prefix in ["Regulation_", "Law_", "Guide_", "Paper_"]:
        if name_no_ext.startswith(prefix):
            name_no_ext = name_no_ext[len(prefix):]
            break
    spaced = name_no_ext.replace("_", " ")
    return turkish_title(spaced)


In [13]:
def process_pdfs_in_directory(pdf_folder: str, source_label: str, batch_size: int = 5, output_dir: str | None = None):
    qa_agent = QA_Agent("QA_Agent", QA_role)
    qa_agent.reset_costs()

    if output_dir is not None and not os.path.exists(output_dir):
        os.makedirs(output_dir, exist_ok=True)

    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

    for pdf_name in pdf_files:
        qa_list = []
        pdf_path = os.path.join(pdf_folder, pdf_name)
        print(f"Processing PDF: {pdf_name}")
        derived_source = source_from_filename(pdf_name)

        # PDF'i madde bazlı metin birimlerine çevir
        units = convert_pdf_to_articles(pdf_path)
        print("\nExtracted units:")
        for i, unit in enumerate(units):
            print(f"\nUnit {i+1}:")
            print(unit)
            print("-" * 80)

        csv_name = pdf_name.split(".")[0] + ".csv"

        start_idx = 0
        end_idx = len(units)

        for i in range(start_idx, end_idx, batch_size):
            # Metin birimlerini (madde/parça) batch halinde işle
            batch_units = units[i : min(i + batch_size, end_idx)]
            print(f"\nBatch {i // batch_size + 1} started")
            print(f"Processing units {i+1} to {min(i + batch_size, end_idx)}")

            for unit in batch_units:
                print(f"\nProcessing unit:\n{unit}\n")
                while True:  # Retry loop
                    # Determine text - model will decide how many questions to generate
                    unit_text = unit["context"] if isinstance(unit, dict) and "context" in unit else unit

                    generated_QA = generate_QA(qa_agent, unit_text)

                    if generated_QA is None:
                        # Hata mesajı kota aşımıysa bekle
                        print("Quota exceeded or generation failed, waiting before retry...")
                        time.sleep(10)
                        continue

                    try:
                        # JSON'u temizle ve parse et
                        cleaned = _extract_json_array(generated_QA) or generated_QA
                        generated_QA_JSON = json.loads(cleaned)
                        # Standart context/source alanlarını garanti altına al
                        for item in generated_QA_JSON:
                            if not item.get("context"):
                                item["context"] = unit_text
                            # Always override source from filename to ensure consistency
                            item["source"] = derived_source
                        print(f"Generated {len(generated_QA_JSON)} questions for this unit")
                        break
                    except json.JSONDecodeError as e:
                        print(f"Error decoding JSON: {e}")
                        print(f"Problematic string: {generated_QA}")
                        print("Retrying...")
                        time.sleep(5)

                qa_list.extend(generated_QA_JSON)
                print("a unit in the batch processed...")

            print(f"Batch {i // batch_size + 1} finished...")
            qa_agent.cost()

            # Batch sonuçlarını kaydet
            out_prefix = f"{csv_name}batch{i // batch_size + 1}"
            if output_dir is not None:
                out_prefix = os.path.join(output_dir, out_prefix)
            save_csv(out_prefix, qa_list)
            print(f"{out_prefix}.csv saved...")
            qa_list = []  # Sonraki batch için sıfırla

        qa_agent.cost()
        print(f"Finished PDF: {pdf_name}")

In [14]:
"""def convert_pdf_to_text(pdf_name):
    doc = pymupdf.open(pdf_name)
    print(f"pdf has {len(doc)} pages")

    page_texts = []
    for page in doc:
        page_texts.append(page.get_text())

    return page_texts"""

'def convert_pdf_to_text(pdf_name):\n    doc = pymupdf.open(pdf_name)\n    print(f"pdf has {len(doc)} pages")\n\n    page_texts = []\n    for page in doc:\n        page_texts.append(page.get_text())\n\n    return page_texts'

In [15]:
"""if __name__ == "__main__":
    qa_agent = QA_Agent("QA_Agent", QA_role)
    qa_agent.reset_costs()
    batch_size = 5

    # PDF dosyalarının bulunduğu klasör
    pdf_folder = "/Users/beyzaasan/Projects/bitirme-projesi/belge-hukukPusulasi-veri/Law"  
    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]

    for pdf_name in pdf_files:
        qa_list = []
        pdf_path = os.path.join(pdf_folder, pdf_name)
        print(f"Processing PDF: {pdf_name}")

        # PDF'i madde bazlı metin birimlerine çevir
        units = convert_pdf_to_articles(pdf_path)
        print("\nExtracted units:")
        for i, unit in enumerate(units):
            print(f"\nUnit {i+1}:")
            print(unit)
            print("-" * 80)

        csv_name = pdf_name.split(".")[0] + ".csv"

        start_idx = 0
        end_idx = len(units)

        for i in range(start_idx, end_idx, batch_size):
            # Metin birimlerini (madde/parça) batch halinde işle
            batch_units = units[i : min(i + batch_size, end_idx)]
            print(f"\nBatch {i // batch_size + 1} started")
            print(f"Processing units {i+1} to {min(i + batch_size, end_idx)}")

            for unit in batch_units:
                print(f"\nProcessing unit:\n{unit}\n")
                while True:  # Retry loop
                    # Determine text - model will decide how many questions to generate
                    unit_text = unit["context"] if isinstance(unit, dict) and "context" in unit else unit

                    generated_QA = generate_QA(qa_agent, unit_text)

                    if generated_QA is None:
                        # Hata mesajı kota aşımıysa bekle
                        print("Quota exceeded or generation failed, waiting before retry...")
                        time.sleep(10)
                        continue

                    try:
                        # JSON'u temizle ve parse et
                        cleaned = _extract_json_array(generated_QA) or generated_QA
                        # Yarım kalan JSON'u düzelt
                        #cleaned = _fix_incomplete_json(cleaned)
                        generated_QA_JSON = json.loads(cleaned)
                        # Standart context/source alanlarını garanti altına al
                        for item in generated_QA_JSON:
                            if not item.get("context"):
                                item["context"] = unit_text
                            if not item.get("source"):
                                item["source"] = "TÜKETİCİNİN KORUNMASI HAKKINDA KANUN"
                            # Model zaten article_id ve article_ref'yi doğru formatta üretiyor
                            # Eğer eksikse, sadece uyarı ver ama kodla ekleme
                            if not item.get("article_id"):
                                print(f"Warning: Missing article_id in generated Q/A")
                            if not item.get("article_ref"):
                                print(f"Warning: Missing article_ref in generated Q/A")
                        print(f"Generated {len(generated_QA_JSON)} questions for this unit")
                        break
                    except json.JSONDecodeError as e:
                        print(f"Error decoding JSON: {e}")
                        print(f"Problematic string: {generated_QA}")
                        print("Retrying...")
                        time.sleep(5)

                qa_list.extend(generated_QA_JSON)
                print("a unit in the batch processed...")

            print(f"Batch {i // batch_size + 1} finished...")
            qa_agent.cost()

            # Batch sonuçlarını kaydet
            save_csv(f"{csv_name}batch{i // batch_size + 1}", qa_list)
            print(f"{csv_name}batch{i // batch_size + 1} saved...")
            qa_list = []  # Sonraki batch için sıfırla

        qa_agent.cost()
        print(f"Finished PDF: {pdf_name}")"""

'if __name__ == "__main__":\n    qa_agent = QA_Agent("QA_Agent", QA_role)\n    qa_agent.reset_costs()\n    batch_size = 5\n\n    # PDF dosyalarının bulunduğu klasör\n    pdf_folder = "/Users/beyzaasan/Projects/bitirme-projesi/belge-hukukPusulasi-veri/Law"  \n    pdf_files = [f for f in os.listdir(pdf_folder) if f.endswith(".pdf")]\n\n    for pdf_name in pdf_files:\n        qa_list = []\n        pdf_path = os.path.join(pdf_folder, pdf_name)\n        print(f"Processing PDF: {pdf_name}")\n\n        # PDF\'i madde bazlı metin birimlerine çevir\n        units = convert_pdf_to_articles(pdf_path)\n        print("\nExtracted units:")\n        for i, unit in enumerate(units):\n            print(f"\nUnit {i+1}:")\n            print(unit)\n            print("-" * 80)\n\n        csv_name = pdf_name.split(".")[0] + ".csv"\n\n        start_idx = 0\n        end_idx = len(units)\n\n        for i in range(start_idx, end_idx, batch_size):\n            # Metin birimlerini (madde/parça) batch halinde işle

In [ ]:
# Run processing with category-specific outputs and batch sizes
law_dir = "/Users/beyzaasan/Projects/bitirme-projesi/hukukPusulasi-veri/Law"
reg_dir = "/Users/beyzaasan/Projects/bitirme-projesi/hukukPusulasi-veri/Regulation"
paper_dir = "/Users/beyzaasan/Projects/bitirme-projesi/hukukPusulasi-veri/Paper"
guide_dir = "/Users/beyzaasan/Projects/bitirme-projesi/hukukPusulasi-veri/Guide"

# Output roots per category
out_root = "/Users/beyzaasan/Projects/bitirme-projesi/outputs"
law_out = os.path.join(out_root, "Law")
reg_out = os.path.join(out_root, "Regulation")
paper_out = os.path.join(out_root, "Paper")
guide_out = os.path.join(out_root, "Guide")

# Batch sizes per category
batch_law = 5
batch_reg = 5
batch_paper = 7
batch_guide = 9

# Kanun (isteğe bağlı çalıştır)
# process_pdfs_in_directory(law_dir, source_label="TÜKETİCİNİN KORUNMASI HAKKINDA KANUN", batch_size=batch_law, output_dir=law_out)

# Yönetmelik
# process_pdfs_in_directory(reg_dir, source_label="YÖNETMELİK", batch_size=batch_reg, output_dir=reg_out)

# Tebliğ/Makale
process_pdfs_in_directory(paper_dir, source_label="TEBLİĞ", batch_size=batch_paper, output_dir=paper_out)

# Kılavuz
# process_pdfs_in_directory(guide_dir, source_label="KILAVUZ", batch_size=batch_guide, output_dir=guide_out)

Processing PDF: Paper_6502_SAYILI_TÜKETİCİNİN_KORUNMASI_HAKKINDA_KANUNUN_77_NCİ_MADDESİNE_GÖRE_2025_YILINDA_UYGULANACAK_OLAN_İDARİ_PARA_CEZALARINA_İLİŞKİN_TEBLİĞ.pdf
articles extracted: 5, subclauses/items: 5, units: 5

Extracted units:

Unit 1:
MADDE 1- (1) Bu Tebliğin amacı, 7/11/2013 tarihli ve 6502 sayılı Tüketicinin Korunması 
Hakkında Kanunun 77 nci maddesinde düzenlenmiş olan idari para cezalarının, 27/11/2024 tarihli ve 
32735 sayılı Resmî Gazete’de yayımlanan Vergi Usul Kanunu Genel Tebliği (Sıra No: 574)’nde 2024 yılı 
için yeniden değerleme oranı olarak tespit edilen % 43,93 (yüzde kırk üç virgül doksan üç) oranında 
artırılarak yeniden belirlenmesidir. 
Dayanak
--------------------------------------------------------------------------------

Unit 2:
MADDE 2- (1) Bu Tebliğ, 7/11/2013 tarihli ve 6502 sayılı Tüketicinin Korunması Hakkında 
Kanunun 77 nci maddesi ile 84 üncü maddesinin birinci fıkrasına ve 30/3/2005 tarihli ve 5326 sayılı 
Kabahatler Kanununun 17

In [17]:
# Combine CSVs from category-specific output folders
out_root = "/Users/beyzaasan/Projects/bitirme-projesi/outputs"
category_dirs = {
    "Law": os.path.join(out_root, "Law"),
    "Regulation": os.path.join(out_root, "Regulation"),
    "Guide": os.path.join(out_root, "Guide"),
    "Paper": os.path.join(out_root, "Paper"),
}

combined_outputs_dir = "/Users/beyzaasan/Projects/bitirme-projesi/outputs"

def combine_folder_csvs(folder_path):
    files = glob.glob(os.path.join(folder_path, "*.csv"))
    if not files:
        return None
    frames = []
    for file in files:
        try:
            frames.append(pd.read_csv(file))
        except pd.errors.EmptyDataError:
            print(f"Warning: Skipping empty file: {file}")
        except pd.errors.ParserError:
            print(f"Warning: Skipping file with parsing errors: {file}")
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)

# Per-category combine
all_frames = []
for name, path in category_dirs.items():
    if not os.path.isdir(path):
        print(f"Skip missing category folder: {name}")
        continue
    df_cat = combine_folder_csvs(path)
    if df_cat is None:
        print(f"No CSVs to combine for {name}")
        continue
    out_path = os.path.join(combined_outputs_dir, f"{name}_combined.csv")
    df_cat.to_csv(out_path, index=False)
    print(f"Wrote {name} combined -> {out_path} ({len(df_cat)} rows)")
    all_frames.append(df_cat)

# Overall combine
if all_frames:
    df_all = pd.concat(all_frames, ignore_index=True)
    out_all = os.path.join(combined_outputs_dir, "ALL_categories_combined.csv")
    df_all.to_csv(out_all, index=False)
    print(f"Wrote ALL categories combined -> {out_all} ({len(df_all)} rows)")
else:
    print("No category data found to combine.")

Skip missing category folder: Law
Skip missing category folder: Regulation
Skip missing category folder: Guide
Wrote Paper combined -> /Users/beyzaasan/Projects/bitirme-projesi/outputs/Paper_combined.csv (187 rows)
Wrote ALL categories combined -> /Users/beyzaasan/Projects/bitirme-projesi/outputs/ALL_categories_combined.csv (187 rows)
